# MVP de Engenharia de Dados — Produção de Café no Brasil

## 01 — Ingestão e Camada Bronze

### Objetivo
Este notebook tem como objetivo realizar a ingestão dos dados brutos utilizados no MVP de Engenharia de Dados, preservando as características originais da fonte e armazenando-os na camada Bronze da arquitetura medalhão.

### Fonte dos dados
- **Fonte:** IBGE — Instituto Brasileiro de Geografia e Estatística
- **Pesquisa:** Produção Agrícola Municipal (PAM)
- **Tabela SIDRA:** 1613
- **Tema:** Produção de café no Brasil
- **Período selecionado:** 2015 a 2025
- **Nível territorial:** Unidade da Federação
- **Formato de origem:** CSV

### Produtos selecionados
- Café (em grão) Total
- Café (em grão) Arábica
- Café (em grão) Canephora

### Variáveis selecionadas
- Área destinada à colheita (hectares)
- Área colhida (hectares)
- Quantidade produzida (toneladas)
- Rendimento médio da produção (kg/hectare)
- Valor da produção (mil reais)

### Arquitetura
O projeto utilizará a arquitetura medalhão:

**Bronze → Silver → Gold**

Nesta etapa, os dados serão armazenados na camada **Bronze**, mantendo as características da fonte original. As operações de limpeza, padronização e transformação serão realizadas posteriormente na camada Silver.


## Criação da camada Bronze

Nesta etapa é criado o schema `bronze` no catálogo `workspace`. Esse schema será utilizado para armazenar os dados provenientes da fonte original antes das etapas de limpeza, padronização e transformação.

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Camada Bronze do MVP de Engenharia de Dados - dados brutos provenientes das fontes de origem';

## Armazenamento do arquivo de origem

Para garantir a rastreabilidade dos dados, o arquivo CSV obtido do SIDRA/IBGE será armazenado em um Volume do Unity Catalog.

O arquivo será mantido em seu formato original, sem alterações manuais. As etapas de limpeza, tratamento de valores especiais, padronização e transformação serão realizadas posteriormente no pipeline.

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS workspace.bronze.dados_brutos
COMMENT 'Volume destinado ao armazenamento dos arquivos brutos utilizados no MVP';

## Validação do arquivo de origem

Após o upload do arquivo original para o Volume da camada Bronze, é realizada uma verificação para confirmar sua disponibilidade no armazenamento antes do início da leitura e processamento dos dados.

O arquivo permanece preservado em seu formato CSV original, garantindo rastreabilidade em relação à fonte de dados.

In [0]:
caminho_volume = "/Volumes/workspace/bronze/dados_brutos/"

arquivos = dbutils.fs.ls(caminho_volume)

display(arquivos)

## Inspeção inicial do arquivo bruto

Antes da ingestão dos dados em uma tabela Bronze, é realizada uma inspeção do conteúdo do arquivo CSV para compreender sua estrutura original.

Essa verificação permite identificar características como cabeçalhos, separadores, organização das variáveis e possíveis linhas adicionais fornecidas pela fonte, sem realizar alterações no arquivo original.

In [0]:
caminho_arquivo = "/Volumes/workspace/bronze/dados_brutos/ibge_pam_cafe_2015_2025.csv"

linhas_brutas = spark.read.text(caminho_arquivo)

display(linhas_brutas.limit(15))

## Persistência dos dados na camada Bronze

Após a inspeção da estrutura do arquivo, foi identificado que o CSV exportado pelo SIDRA possui cabeçalhos hierárquicos e blocos correspondentes às diferentes variáveis selecionadas.

Para preservar a estrutura recebida da fonte, os registros serão inicialmente armazenados na camada Bronze em formato textual, sem aplicação de regras de limpeza ou transformação estrutural.

A interpretação dos blocos, tratamento dos valores especiais, padronização e tipagem serão realizados posteriormente na camada Silver.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

bronze_ibge_cafe = (
    spark.read
         .text(caminho_arquivo)
         .withColumnRenamed("value", "linha_original")
)

display(bronze_ibge_cafe.limit(10))

## Gravação da tabela Bronze em formato Delta

Após a validação da leitura do arquivo, os dados brutos serão persistidos em uma tabela Delta na camada Bronze.

Além do conteúdo original, será adicionada a data e hora de ingestão como metadado técnico, permitindo identificar quando os dados foram incorporados ao pipeline.

Nenhuma regra de limpeza ou transformação dos dados de negócio será aplicada nesta etapa.

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_ibge_cafe = (
    bronze_ibge_cafe
    .withColumn("data_ingestao", current_timestamp())
)

(
    bronze_ibge_cafe.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.bronze.ibge_pam_cafe_raw")
)

print("Tabela Bronze criada com sucesso!")

## Validação da camada Bronze

Após a persistência dos dados em formato Delta, será realizada uma validação da tabela Bronze para confirmar que os registros foram armazenados corretamente.

Nesta etapa serão verificados a quantidade de registros, a estrutura da tabela e uma amostra dos dados persistidos. Essa validação permite garantir que a ingestão foi concluída corretamente antes do início das transformações da camada Silver.

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM workspace.bronze.ibge_pam_cafe_raw;

%md
### Validação da estrutura da tabela Bronze

A tabela Bronze possui 204 registros correspondentes às linhas preservadas do arquivo original.

Nesta etapa será verificada a estrutura da tabela, incluindo os nomes das colunas e seus respectivos tipos de dados, garantindo que o conteúdo bruto e os metadados de ingestão tenham sido armazenados conforme planejado.

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.ibge_pam_cafe_raw;

### Validação do conteúdo persistido

Como etapa final de validação da camada Bronze, será consultada uma amostra diretamente da tabela Delta persistida.

O objetivo é verificar se o conteúdo bruto proveniente do arquivo de origem permanece disponível após a gravação e se o metadado de ingestão foi corretamente associado aos registros.

In [0]:
%sql
SELECT
    linha_original,
    data_ingestao
FROM workspace.bronze.ibge_pam_cafe_raw
LIMIT 10;